In [1]:
"""
=============================================================
MODEL 2 — CNN + BiLSTM + ATTENTION (PyTorch)
=============================================================
Hybrid sequential model for telecom alarm threshold prediction.
Runs on GPU (GTX 1650 Ti) or CPU.

Architecture:
  CNN Encoder    → extracts local hourly patterns
  BiLSTM         → captures sequential dependencies
  Attention      → weights hours by context relevance
  Context MLP    → processes date/vendor/region features
  Fusion + Output → 4 threshold predictions

Requirements:
  pip install torch scikit-learn pandas numpy

Usage:
  python train_model2_cnn_bilstm.py
=============================================================
"""

import os
import time
import json
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Device ────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

DATA_DIR   = './ml_outputs_enriched'
OUTPUT_DIR = './model_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_COLS = [
    'TARGET_threshold_hours',
    'TARGET_threshold_inactive_min',
    'TARGET_threshold_fluctuation',
    'TARGET_each_hour_fluctuation',
]
ID_COLS = ['Cell', 'Alarm_Raised_Date', 'Start_Hour']

CLIP_RANGES = [(2,8), (5,200), (1,7), (1,7)]

# ── Hyperparameters ───────────────────────────────────────────
EPOCHS       = 50
BATCH_SIZE   = 1024
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 8       # early stopping patience
GRAD_CLIP    = 1.0

# ── Load data ─────────────────────────────────────────────────
print("\nLoading data...")
X_train_df = pd.read_csv(f'{DATA_DIR}/X_train.csv')
X_test_df  = pd.read_csv(f'{DATA_DIR}/X_test.csv')
y_train_df = pd.read_csv(f'{DATA_DIR}/y_train.csv')
y_test_df  = pd.read_csv(f'{DATA_DIR}/y_test.csv')

FEAT_COLS = [c for c in X_train_df.columns if c not in ID_COLS]

for col in FEAT_COLS:
    if X_train_df[col].isnull().any():
        m = X_train_df[col].median()
        X_train_df[col] = X_train_df[col].fillna(m)
        X_test_df[col]  = X_test_df[col].fillna(m)

# Identify hourly sequence features and context features
hourly_inactive_cols = [f'Hour_{h}_Inactive'    for h in range(1, 25)]
hourly_fluct_cols    = [f'Hour_{h}_Fluctuation' for h in range(1, 25)]
sequence_cols        = hourly_inactive_cols + hourly_fluct_cols
context_cols         = [c for c in FEAT_COLS if c not in sequence_cols]

print(f"  Sequence features: {len(sequence_cols)} (24 inactive + 24 fluct)")
print(f"  Context features:  {len(context_cols)}")

# ── Dataset ───────────────────────────────────────────────────
class AlarmDataset(Dataset):
    def __init__(self, X_df, y_df):
        # Sequence: (N, 24, 2) — stack inactive and fluct per hour
        inactive = X_df[hourly_inactive_cols].values.astype(np.float32)
        fluct    = X_df[hourly_fluct_cols].values.astype(np.float32)
        self.seq = torch.tensor(
            np.stack([inactive, fluct], axis=2)  # (N, 24, 2)
        )
        self.ctx = torch.tensor(
            X_df[context_cols].values.astype(np.float32)
        )
        self.y = torch.tensor(
            y_df[TARGET_COLS].values.astype(np.float32)
        )

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.seq[idx], self.ctx[idx], self.y[idx]


train_dataset = AlarmDataset(X_train_df, y_train_df)
test_dataset  = AlarmDataset(X_test_df,  y_test_df)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE*2,
                          shuffle=False, num_workers=0, pin_memory=True)

print(f"  Train batches: {len(train_loader)}")

# ── Model Architecture ────────────────────────────────────────
class CNNBiLSTMAttention(nn.Module):
    def __init__(self, seq_len=24, n_seq_feat=2, n_ctx=len(context_cols),
                 cnn_channels=64, lstm_hidden=128, lstm_layers=2,
                 attn_heads=4, dropout=0.2, output_dim=4):
        super().__init__()

        self.seq_len     = seq_len
        self.lstm_hidden = lstm_hidden

        # ── CNN Encoder ───────────────────────────────────────
        self.cnn = nn.Sequential(
            nn.Conv1d(n_seq_feat, cnn_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(),
            nn.Conv1d(cnn_channels, cnn_channels*2, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_channels*2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        cnn_out_channels = cnn_channels * 2   # 128

        # ── BiLSTM ────────────────────────────────────────────
        self.bilstm = nn.LSTM(
            input_size=cnn_out_channels,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0,
        )
        lstm_out_dim = lstm_hidden * 2  # 256 (bidirectional)

        # ── Multi-Head Attention ──────────────────────────────
        self.attention = nn.MultiheadAttention(
            embed_dim=lstm_out_dim,
            num_heads=attn_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.attn_norm = nn.LayerNorm(lstm_out_dim)

        # Query projection from context
        self.query_proj = nn.Linear(n_ctx, lstm_out_dim)

        # ── Context MLP ───────────────────────────────────────
        self.ctx_mlp = nn.Sequential(
            nn.Linear(n_ctx, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # ── Fusion + Output ───────────────────────────────────
        fused_dim = lstm_out_dim + 128  # 256 + 128 = 384
        self.fusion = nn.Sequential(
            nn.Linear(fused_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
        )
        self.output_head = nn.Linear(128, output_dim)

    def forward(self, seq, ctx):
        """
        seq: (B, 24, 2)    — hourly inactive + fluctuation
        ctx: (B, n_ctx)    — context features
        """
        B = seq.size(0)

        # CNN: expects (B, channels, seq_len)
        x = seq.permute(0, 2, 1)             # (B, 2, 24)
        x = self.cnn(x)                       # (B, 128, 24)
        x = x.permute(0, 2, 1)               # (B, 24, 128)

        # BiLSTM
        lstm_out, _ = self.bilstm(x)          # (B, 24, 256)

        # Attention: context vector as query
        query = self.query_proj(ctx)           # (B, 256)
        query = query.unsqueeze(1)             # (B, 1, 256)
        attn_out, attn_weights = self.attention(
            query, lstm_out, lstm_out
        )                                      # (B, 1, 256)
        attn_out = attn_out.squeeze(1)         # (B, 256)
        attn_out = self.attn_norm(attn_out + query.squeeze(1))

        # Context MLP
        ctx_out = self.ctx_mlp(ctx)            # (B, 128)

        # Fusion
        fused = torch.cat([attn_out, ctx_out], dim=1)  # (B, 384)
        fused = self.fusion(fused)                      # (B, 128)
        out   = self.output_head(fused)                 # (B, 4)

        return out, attn_weights


# ── Loss: Huber ───────────────────────────────────────────────
criterion = nn.HuberLoss(delta=1.0)

# ── Model init ────────────────────────────────────────────────
model = CNNBiLSTMAttention(
    n_ctx=len(context_cols),
    cnn_channels=64,
    lstm_hidden=128,
    lstm_layers=2,
    attn_heads=4,
    dropout=0.2,
    output_dim=4,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel parameters: {total_params:,}")

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# ── Training loop ─────────────────────────────────────────────
print(f"\nTraining CNN-BiLSTM-Attention for {EPOCHS} epochs...")
print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>10} {'LR':>10}")
print("-" * 45)

best_val_loss   = float('inf')
patience_count  = 0
best_state      = None
train_losses    = []
val_losses      = []
t_start         = time.time()

for epoch in range(1, EPOCHS + 1):
    # ── Train ────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for seq, ctx, y in train_loader:
        seq, ctx, y = seq.to(DEVICE), ctx.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        pred, _ = model(seq, ctx)
        loss = criterion(pred, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # ── Validate ─────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for seq, ctx, y in test_loader:
            seq, ctx, y = seq.to(DEVICE), ctx.to(DEVICE), y.to(DEVICE)
            pred, _ = model(seq, ctx)
            val_loss += criterion(pred, y).item()
    val_loss /= len(test_loader)

    scheduler.step()
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if epoch % 5 == 0 or epoch == 1:
        lr_now = optimizer.param_groups[0]['lr']
        print(f"  {epoch:4d}   {train_loss:>10.4f}   {val_loss:>8.4f}   {lr_now:>8.6f}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        patience_count = 0
        best_state     = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch}")
            break

train_time = time.time() - t_start
print(f"\nTraining time: {train_time:.1f}s ({train_time/60:.1f} min)")

# Load best weights
model.load_state_dict(best_state)

# ── Evaluate ──────────────────────────────────────────────────
print("\nEvaluating...")
model.eval()
all_preds = []
with torch.no_grad():
    for seq, ctx, y in test_loader:
        seq, ctx = seq.to(DEVICE), ctx.to(DEVICE)
        pred, _ = model(seq, ctx)
        all_preds.append(pred.cpu().numpy())

y_pred = np.vstack(all_preds)
y_true = y_test_df[TARGET_COLS].values.astype(np.float32)

# Clip to valid ranges
for i, (lo, hi) in enumerate(CLIP_RANGES):
    y_pred[:, i] = np.clip(y_pred[:, i], lo, hi)

print(f"\n{'Target':<35} {'MAE':>8} {'RMSE':>8} {'R²':>8}")
print("-" * 62)
results = {}
all_mae, all_rmse, all_r2 = [], [], []
for i, col in enumerate(TARGET_COLS):
    mae  = mean_absolute_error(y_true[:, i], y_pred[:, i])
    rmse = np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))
    r2   = r2_score(y_true[:, i], y_pred[:, i])
    short = col.replace('TARGET_', '')
    print(f"  {short:<33} {mae:>8.4f} {rmse:>8.4f} {r2:>8.4f}")
    results[col] = {'MAE': round(mae,4), 'RMSE': round(rmse,4), 'R2': round(r2,4)}
    all_mae.append(mae); all_rmse.append(rmse); all_r2.append(r2)

print("-" * 62)
print(f"  {'AVERAGE':<33} "
      f"{np.mean(all_mae):>8.4f} "
      f"{np.mean(all_rmse):>8.4f} "
      f"{np.mean(all_r2):>8.4f}")

# ── Save ──────────────────────────────────────────────────────
torch.save({
    'model_state_dict': best_state,
    'model_config': {
        'n_ctx': len(context_cols),
        'cnn_channels': 64,
        'lstm_hidden': 128,
        'lstm_layers': 2,
        'attn_heads': 4,
        'dropout': 0.2,
        'output_dim': 4,
    },
    'sequence_cols': sequence_cols,
    'context_cols':  context_cols,
    'feat_cols':     FEAT_COLS,
    'target_cols':   TARGET_COLS,
}, f'{OUTPUT_DIR}/model2_cnn_bilstm.pt')

pd.DataFrame(y_pred, columns=[c+'_pred' for c in TARGET_COLS]).to_csv(
    f'{OUTPUT_DIR}/model2_predictions.csv', index=False)

summary = {
    'model': 'CNN_BiLSTM_Attention',
    'train_time_sec': round(train_time, 2),
    'total_params': total_params,
    'best_val_loss': round(best_val_loss, 4),
    'avg_MAE':  round(np.mean(all_mae), 4),
    'avg_RMSE': round(np.mean(all_rmse), 4),
    'avg_R2':   round(np.mean(all_r2), 4),
    'per_target': results,
    'train_losses': train_losses,
    'val_losses':   val_losses,
}
with open(f'{OUTPUT_DIR}/model2_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✅ Model 2 saved: model2_cnn_bilstm.pt")
print(f"   Predictions:   model2_predictions.csv")
print(f"   Summary:       model2_summary.json")

# ── Fine-tune function for daily incremental update ───────────
def fine_tune(model_path, X_new_df, y_new_df, epochs=5, device=DEVICE):
    """
    Fine-tune on new day's data.
    Call this each day after new 24hr_summary.csv is processed.
    """
    checkpoint = torch.load(model_path, map_location=device)
    m = CNNBiLSTMAttention(**checkpoint['model_config']).to(device)
    m.load_state_dict(checkpoint['model_state_dict'])

    new_dataset = AlarmDataset(X_new_df, y_new_df)
    new_loader  = DataLoader(new_dataset, batch_size=512, shuffle=True)
    opt = optim.AdamW(m.parameters(), lr=1e-4)
    crit = nn.HuberLoss(delta=1.0)

    m.train()
    for ep in range(epochs):
        for seq, ctx, y in new_loader:
            seq, ctx, y = seq.to(device), ctx.to(device), y.to(device)
            opt.zero_grad()
            pred, _ = m(seq, ctx)
            crit(pred, y).backward()
            opt.step()

    torch.save({**checkpoint, 'model_state_dict': m.state_dict()}, model_path)
    print(f"Fine-tuned for {epochs} epochs on {len(new_dataset)} new samples.")

Using device: cuda
  GPU: NVIDIA GeForce GTX 1650 Ti
  VRAM: 4.3 GB

Loading data...
  Sequence features: 48 (24 inactive + 24 fluct)
  Context features:  75
  Train batches: 198

Model parameters: 1,153,732

Training CNN-BiLSTM-Attention for 50 epochs...
 Epoch   Train Loss   Val Loss         LR
---------------------------------------------
     1       1.4051     0.6992   0.000999
     5       0.8226     0.6586   0.000976
    10       0.4771     0.4592   0.000905
    15       0.3268     0.3756   0.000794
    20       0.2709     0.3951   0.000655

  Early stopping at epoch 22

Training time: 778.0s (13.0 min)

Evaluating...

Target                                   MAE     RMSE       R²
--------------------------------------------------------------
  threshold_hours                     0.2361   0.4439   0.5915
  threshold_inactive_min              1.3773   7.4073   0.6971
  threshold_fluctuation               0.0838   0.2598   0.0080
  each_hour_fluctuation               0.0734   0.26